<a href="https://www.kaggle.com/code/ahmedfakhar123/ml-19-columntransformer-in-scikit-learn?scriptVersionId=340156735" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

---

# 🤖 ML 19 — ColumnTransformer in Scikit-learn

### Apply different preprocessing techniques to different columns with ColumnTransformer

---

# 🚀 Introduction

In real-world datasets, different columns often require different preprocessing techniques. For example, one column may need **One-Hot Encoding**, another may require **Ordinal Encoding**, while numerical features might need **Standardization** or **Normalization**.

Applying these transformations manually can be repetitive and difficult to manage, especially as datasets grow larger.

**ColumnTransformer** in **Scikit-learn** solves this problem by allowing you to apply different transformations to different columns in a single, organized preprocessing pipeline. It keeps your workflow clean, efficient, and easy to integrate with Machine Learning models.

In this notebook, you'll learn how to use **ColumnTransformer** to preprocess mixed datasets, combine multiple transformers, and build scalable machine learning workflows with practical examples. 🚀

<div align="center">
    <img src="https://media.geeksforgeeks.org/wp-content/uploads/20240802120358/Withoutcolumntransformer.webp" width=500>
    <img src="https://media.geeksforgeeks.org/wp-content/uploads/20240802120607/usingcolumntransformer.webp" width=500>
    <img src="https://media.geeksforgeeks.org/wp-content/uploads/20240802120842/Single-Vector-containing-different-feature--values.webp" width=500>
</div>

---

# 🤔 What is ColumnTransformer?

**ColumnTransformer** is a Scikit-learn preprocessing tool that allows us to apply **different transformations to different columns** of the same dataset.

Instead of preprocessing columns separately, we can combine everything into one object.


# ❓ Why do we need ColumnTransformer?

Imagine a dataset like this:

| Age | Salary | Gender | Education |
| --- | ------ | ------ | --------- |
| 25  | 50000  | Male   | Bachelor  |

Different columns require different preprocessing.

| Column    | Transformation |
| --------- | -------------- |
| Age       | StandardScaler |
| Salary    | StandardScaler |
| Gender    | OneHotEncoder  |
| Education | OrdinalEncoder |

Without ColumnTransformer:

* preprocess every column separately
* combine manually
* difficult to maintain

With ColumnTransformer:

Everything happens inside one transformer.

---

# 📦 Importing Libraries



In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder
)

---

# 📊 Reading Dataset

In [2]:
df = pd.read_csv("/kaggle/input/datasets/ahmedfakhar123/customer-purchase-prediction-dataset/Customer_Encoding_Dataset.csv")
df

,age,gender,review,education,purchased
0,38,Male,Average,Diploma,No
1,24,Female,Good,Bachelor,Yes
2,23,Female,Average,Diploma,No
3,21,Male,Excellent,Bachelor,Yes
4,21,Female,Poor,Diploma,No
...,...,...,...,...,...
115,49,Female,Poor,High School,No
116,46,Female,Average,Diploma,No
117,38,Male,Good,Diploma,Yes
118,43,Male,Excellent,Diploma,No


---

# 👀 Understanding the Dataset

We have

✅ Numerical Columns

* Age
* Salary

✅ Nominal Column

* Gender

✅ Ordinal Column

* Education

Education has an order:

```
Diploma
↓

Bachelor
↓

Master
↓

PhD
```



---

# 🔤 Step 1 — One-Hot Encoding

We'll apply **OneHotEncoder** to

* Gender


## 1.1 Seperating Features and Target

In [3]:
X = df.drop(columns=["purchased"])
y = df.purchased

## 2.2 Train Test Split

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

## 2.2 Encoding Nominal Data

In [5]:
one_hot_encoder = OneHotEncoder(drop="first", sparse_output=False)

X_train_gender = one_hot_encoder.fit_transform(X_train[["gender"]])
X_test_gender = one_hot_encoder.transform(X_test[["gender"]])

X_train_gender[:20]

array([[1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.]])

---

# 📊 Step 2 — Ordinal Encoding

We'll apply

In [6]:
review_categories = ["Poor", "Average", "Good", "Excellent"]
education_categories = [
    "School", "High School", "Diploma",
    "Bachelor", "Master", "PhD"
]

ordinal_encoder = OrdinalEncoder(categories=[review_categories, education_categories])

X_train_ordinal = ordinal_encoder.fit_transform(
    X_train[['review', 'education']]
)

X_test_ordinal = ordinal_encoder.transform(
    X_test[['review', 'education']]
)

In [7]:
# Extracting Age
X_train_age = X_train[["age"]].values

# also the test data
X_test_age = X_test[["age"]].values

X_train_age.shape

(90, 1)

In [8]:
X_train_transformed = np.concat((X_train_age, X_train_gender, X_train_ordinal), axis=1)
# also the test data
X_test_transformed = np.concat((X_test_age, X_test_gender, X_test_ordinal), axis=1)

X_train_transformed.shape

(90, 4)

---
# 🏗️ Creating ColumnTransformer

So far, we've applied different preprocessing techniques separately and manually combined the transformed features. While this approach works, it quickly becomes **lengthy, repetitive, and difficult to maintain**, especially when working with larger datasets containing many columns.

Instead of manually encoding, scaling, extracting columns, and concatenating the results, **ColumnTransformer** lets us apply multiple transformations to different columns in a single, clean, and organized step.

Let's simplify our preprocessing workflow by creating a **ColumnTransformer**.

In [9]:
transformer = ColumnTransformer([
    ("ordinal_encoder",
     OrdinalEncoder(categories=[review_categories, education_categories]),
     ["review", "education"]),

    ("onehot_encoder",
     OneHotEncoder(sparse_output=False, drop="first"),
     ["gender"])
], remainder="passthrough")

---

# 🔄 Transforming the Dataset

In [10]:
X_train_transformed = transformer.fit_transform(X_train)

In [11]:
X_test_transformed = transformer.transform(X_test)

---

# 📋 Converting Output to DataFrame

In [12]:
columns = ["review", "education", "male", "age"]

pd.DataFrame(X_train_transformed, columns=columns)

,review,education,male,age
0,0.0,3.0,1.0,30.0
1,0.0,3.0,1.0,55.0
2,0.0,2.0,1.0,54.0
3,2.0,4.0,1.0,48.0
4,3.0,3.0,1.0,21.0
...,...,...,...,...
85,2.0,5.0,1.0,32.0
86,2.0,4.0,0.0,60.0
87,2.0,2.0,1.0,38.0
88,0.0,3.0,1.0,28.0


---

# 🔍 Understanding the Output

You will notice:

* Gender becomes multiple binary columns.
* Education becomes numerical.
* Age and Salary become standardized.
* Everything is combined into one dataset.

Example

| gender_Female | gender_Male | education | Age | Salary |
| ------------- | ----------- | --------- | --- | ------ |

This transformed dataset is now ready for Machine Learning.

---

# 🎯 Advantages of ColumnTransformer

✅ Cleaner code

✅ Applies multiple preprocessing techniques together

✅ Avoids manual concatenation

✅ Easy to integrate with Pipelines

✅ Perfect for production Machine Learning

✅ Reduces preprocessing mistakes

---

# 🎉 Conclusion

Congratulations! 🎉 You've learned how to preprocess mixed datasets using **ColumnTransformer** in Scikit-learn.

In this notebook, you explored how to:

* ✅ Apply **One-Hot Encoding** to nominal features.
* ✅ Apply **Ordinal Encoding** to ordered categories.
* ✅ Scale numerical features using **StandardScaler**.
* ✅ Combine multiple preprocessing steps into a single **ColumnTransformer**.
* ✅ Transform an entire dataset with one clean and reusable workflow.

`ColumnTransformer` is an essential preprocessing tool because it keeps your code organized, reduces manual work, and integrates seamlessly with Scikit-learn Pipelines. Mastering it will make your machine learning workflows more efficient and production-ready. 🚀

---

# 🚀 What's Next?

Great work! 🎉 You've learned how to preprocess different types of features using **ColumnTransformer**.

In many machine learning projects, preprocessing is only one part of the workflow. The next step is to combine preprocessing and model training into a single reusable pipeline.

[**🤖 ML 20 — Pipelines in Scikit-learn**](http://)

You'll learn:

* 🚀 What is a Pipeline?
* 🔗 Why Pipelines are important
* ⚙️ Combining preprocessing and models
* 🤖 Training models with a single workflow
* 🏆 Building clean, scalable, and production-ready machine learning pipelines